# 05. Hyperparameter Tuning — ハイパーパラメータ探索

README の工程 **⑤ Hyperparameter Tuning** にあたる。実行は `src/05_hyperparameter_tuning.py`で行う。

## 05-1. 04 との関係

- `src/05_hyperparameter_tuning.py` は**学習コードを持たない**。`src/04_train_and_evaluate_<model>.py` を引数を変えて1本ずつ呼び、
  OOF AUC を `docs/hyperparameter_tuning_results.csv` に記録するだけ
- 呼ぶ向きは **05 → 04 だけ**。結果が 04 に自動で反映される仕組みはなく、採否は人が DeLong 検定で判断する
- `FIXED`(本番の実行コマンドのうち、探索しない固定部分)が**本番のパラメータの正本**。
  `04_train_and_evaluate.ipynb` はここから値を書き写し、04-6 章で `FIXED` と同じかを照合している
- このノートブックは探索を実行しない(見積もりと、記録済みの結果の表示だけ。数秒で終わる)

## 05-2. ハイパーパラメータの位置づけ

パラメータの調整は一通り Feature Engineering の構成か整った段階から調整していく方向で進めた。




| 施策 | 改善幅 | 種別 |
|---|---|---|
| 厳密値 Target Encoding | +0.003 | FE |
| 収束の確認(lr 引き下げ + early stopping) | +0.0008 | パラメータ |
| Triple TE + digit + ビン数1024 | +0.0005〜0.001 | FE |
| 列サブサンプリング(`colsample=0.3`, `max_depth=5`) | +0.0002 | パラメータ |

FE が桁違いに効いており、パラメータ調整は後から効いた。ただし**列サブサンプリングのように
「見落としていた既定値」は例外的に大きい**。これは探索というより是正だった。

## 05-3. 現在地と期待値

- 最良: CV 0.94623 / **Public LB 0.94645**(**320位 / 2,732チーム、上位11.7%**。2026-09-23 時点の暫定値)
- 目標の上位15%(409位)には**暫定で収まっている**が、参加チームが増え続けるため**確定ではない**
  (9/21 266位 → 9/22 307位 → 9/23 320位と、スコアを変えていないのに下がり続けている)
- 1位は 0.94945、2位は 0.94672。**1位だけ 0.0027 離れた外れ値**で public 過剰最適化の疑いがある

**期待値は低いと見ている。** `research.md` の外部調査によれば、列サブサンプリング導入後に
深さや列比率をさらに振った試行(`d4` / `d6` / `ff02` / `ff04` / `mc30` / `lr01`)は
**すべて -0.000023〜+0.000017** で、誤差の範囲だった。

それでも回す理由は2つ。

1. **LightGBM の `num_leaves` がデフォルトの 31 のまま**で、特徴量が 13 → 92 列に増えている。
   ここだけは未調整の軸が残っている
2. 上位陣が 0.00001 差で入れ替わる状況なので、**誤差に埋もれない改善があるなら拾いたい**

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
# 絶対パスは出力しない(公開リポジトリに個人のディレクトリ構成を残さないため)
print("src/ を検出:", os.path.isdir("src"))

src/ を検出: True


## 05-4. 設計

`src/05_hyperparameter_tuning.py` は**学習コードを持たない**。既存の `04_train_and_evaluate_<model>.py` を引数違いで
呼ぶだけにしてある。理由は、収束設定(lr 引き下げ + early stopping)や FE 構成を本番とズラさないため。

| 方針 | 理由 |
|---|---|
| 既存スクリプトを呼ぶだけ | 本番と同じパイプラインで測る。構成がズレると比較にならない |
| **1本ずつ順番に実行** | 8コア環境で並列にすると CPU を取り合って完走しない(実際に CatBoost が 0.33 コアまで押し出された) |
| **実行前に所要時間を見積もる** | `--estimate` で試行数 × 実測の1本あたり時間を出す |
| 判定は paired DeLong 検定 | AUC の目視では 0.0001 以下の差を判断できない |
| CatBoost は成果物を保存しない | `--save` が本番の成果物を上書きしてしまうため |

## 05-5. 探索する軸

現行ベストと同じ値は含めていない(基準はスクリプト側の `BASELINE` が持っている)。

In [2]:
import importlib
# ファイル名が数字で始まるので、import 文ではなく importlib で読み込む
tuning = importlib.import_module("05_hyperparameter_tuning")
import pandas as pd

rows = []
for model, axes in tuning.SPACE.items():
    for axis, items in axes.items():
        rows.append([model, axis, len(items), ", ".join(n for n, _ in items)])
pd.DataFrame(rows, columns=["モデル", "軸", "試行数", "試行名"])

,モデル,軸,試行数,試行名
0,lgbm,num_leaves,3,"nl15, nl63, nl127"
1,lgbm,feature_fraction,3,"ff02, ff04, ff05"
2,lgbm,max_depth,3,"d4, d6, d7"
3,xgb,max_depth,3,"d4, d6, d7"
4,xgb,colsample,3,"cs02, cs04, cs05"
5,xgb,min_child_weight,2,"mcw5, mcw20"
6,xgb,subsample,2,"sub08, sub06"
7,catboost,depth,3,"d5, d7, d8"
8,catboost,one_hot,2,"ohms16, ohms64"


### 軸を選んだ理由

**LightGBM**
- `num_leaves` — **デフォルト 31 のまま**。特徴量が 92 列に増えているので、木の表現力が足りていない可能性がある。今回の本命
- `feature_fraction` / `max_depth` — 0.3 / 5 が最良かの確認。外部調査では誤差だった

**XGBoost**
- `max_depth` / `colsample_bytree` — 同上
- `min_child_weight` / `subsample` — 未調整の軸。行方向のサンプリングは列方向とは別の効き方をする可能性

**CatBoost**
- `depth` / `one_hot_max_size` — 未調整の軸
- **`rsm`(列サンプリング)は探索しない。** 実測で有害と判明済み(-0.00015〜-0.00039)。
  対称木はすべての深さで同じ分割条件を使うため、列を間引くと木全体が一斉に弱くなる

## 05-6. 所要時間の見積もり

**実行するかどうかは、この見積もりを見てから判断する。**

In [3]:
for model in tuning.SPACE:
    n = len(tuning.build_trials(model))
    per = tuning.RUNTIME[model]
    print(f"{model:9s} {n:2d} 本 × {per/60:4.1f} 分 = 約 {n*per/3600:4.1f} 時間")

lgbm       9 本 ×  4.2 分 = 約  0.6 時間
xgb       10 本 × 10.8 分 = 約  1.8 時間
catboost   5 本 × 25.0 分 = 約  2.1 時間


| モデル | 試行数 | 1本あたり | 合計 | 優先度 |
|---|---|---|---|---|
| LightGBM | 9 | 約 4.2 分 | **約 0.6 時間** | **高**(`num_leaves` が未調整) |
| XGBoost | 10 | 約 10.8 分 | 約 1.8 時間 | 中 |
| CatBoost | 5 | 約 25 分 | 約 2.1 時間 | **低**(現在アンサンブルの重みが 0) |

**合計 約 4.5 時間。** CatBoost が全体の約半分を占めるが、そのCatBoostは現在アンサンブルに
寄与していない(重み 0)。
1本あたりの時間は `src/05_hyperparameter_tuning.py` の `RUNTIME`(実測値)による。CatBoost の実測は 10〜30 分(深さを上げるほど長い)。**まず LightGBM の 0.6 時間だけ回すのが費用対効果が高い。**

## 05-7. 実行

```bash
uv run src/05_hyperparameter_tuning.py lgbm --estimate           # 見積もりだけ
uv run src/05_hyperparameter_tuning.py lgbm                      # 実行(9本、約40分)
uv run src/05_hyperparameter_tuning.py lgbm --only num_leaves    # 本命の軸だけ(3本、約13分)
```

結果は `docs/hyperparameter_tuning_results.csv` に追記される。下のセルはノートブックから直接見積もりを出す例。

In [4]:
trials = tuning.build_trials("lgbm")
tuning.estimate("lgbm", trials)

モデル: lgbm   試行数: 9 本
1本あたり: 約 4.2 分(フル5-fold・7スレッドの実測値)
**合計見積もり: 約 0.6 時間 (38 分)**

軸                  試行名                    引数
------------------------------------------------------------------------------
num_leaves         lgbm_hpo_nl15          --num_leaves 15
num_leaves         lgbm_hpo_nl63          --num_leaves 63
num_leaves         lgbm_hpo_nl127         --num_leaves 127
feature_fraction   lgbm_hpo_ff02          --feature_fraction 0.2
feature_fraction   lgbm_hpo_ff04          --feature_fraction 0.4
feature_fraction   lgbm_hpo_ff05          --feature_fraction 0.5
max_depth          lgbm_hpo_d4            --max_depth 4
max_depth          lgbm_hpo_d6            --max_depth 6
max_depth          lgbm_hpo_d7            --max_depth 7

※ 1本ずつ順番に実行する(並列にすると CPU を取り合って完走しない)
※ 判定は実行後に: uv run src/07_compare_predictions.py <baseline> <tag>


## 05-8. 判定

AUC の数字を見比べるのではなく、**paired DeLong 検定**で有意性を確認する(工程⑦)。

```bash
uv run src/07_compare_predictions.py lgbm --all
```

| | 従来 | DeLong |
|---|---|---|
| ノイズ床(SE) | 0.00015 | **0.00003 前後** |
| 採否基準 | 差分 ≥ +0.0002 | 差分 ≥ +0.00008 **かつ** z ≥ 3 |

**この工程を省略してはいけない。** CV で +0.000009(z=+1.57、有意でない)の構成を提出したところ、
LB では -0.00002 と逆に動いた実例がある。

## 05-9. 結果(2026-09-21〜22 実行)

**計画した 24 試行のうち、実行した 18 試行がすべて誤差または悪化。採用ゼロ。**
LightGBM の `feature_fraction` / `max_depth` の 6 本は、`num_leaves` の結果を見て打ち切った(下記)。
記録(`docs/hyperparameter_tuning_results.csv`)は延べ 21 行ある。`num_leaves` の 3 本を2回実行したため。 判定は paired DeLong(差分 ≥ +0.00008 かつ z ≥ 3)。

### LightGBM(基準 0.946095)

| 軸 | 試行 | OOF AUC | 差分 | z | 判定 |
|---|---|---|---|---|---|
| num_leaves | nl15 | 0.946073 | -0.000022 | -1.65 | 誤差 |
| num_leaves | nl63 | 0.946100 | +0.000004 | +0.42 | 誤差 |
| num_leaves | nl127 | 0.946100 | +0.000004 | +0.42 | 誤差 |

**nl63 と nl127 が小数点以下6桁まで完全一致**したのが決定的だった。原因は **`max_depth=5` が先に
制約になっていた**こと。深さ5の木は葉が最大32枚なので、63 や 127 を指定しても到達しない。
`feature_fraction` / `max_depth` の探索は、この時点で無意味と判断して打ち切った。

### XGBoost(基準 0.946077)

| 軸 | 試行 | OOF AUC | 差分 | z | 判定 |
|---|---|---|---|---|---|
| max_depth | d4 | 0.946080 | +0.000003 | +0.24 | 誤差 |
| max_depth | d6 | 0.945995 | -0.000082 | **-5.77** | **悪化** |
| max_depth | d7 | 0.945966 | -0.000111 | **-6.33** | **悪化** |
| colsample | cs02 | 0.946065 | -0.000012 | -0.89 | 誤差 |
| colsample | cs04 | 0.946064 | -0.000012 | -1.03 | 誤差 |
| colsample | cs05 | 0.946045 | -0.000032 | -2.43 | 誤差 |
| min_child_weight | mcw5 | 0.946079 | +0.000002 | +0.19 | 誤差 |
| min_child_weight | mcw20 | 0.946038 | -0.000039 | -3.00 | 誤差 |
| subsample | sub08 | 0.946081 | +0.000004 | +0.26 | 誤差 |
| subsample | sub06 | 0.946055 | -0.000022 | -1.36 | 誤差 |

**深さを増やすと有意に悪化**(d6 で z=-5.77、d7 で z=-6.33)。現行の `max_depth=5` が最適で、
そこから浅くしても深くしても良くならない。列比率(0.3)と行サンプリングも動かす余地がなかった。

### CatBoost(基準 0.94589)

| 軸 | 試行 | OOF AUC | 差分 | 判定 |
|---|---|---|---|---|
| depth | d5 | 0.945950 | +0.000060 | 誤差(基準の3/4) |
| depth | d7 | 0.945820 | -0.000070 | 悪化 |
| depth | d8 | 0.945650 | -0.000240 | 悪化 |
| one_hot_max_size | 16 | 0.945890 | ±0.000000 | 誤差 |
| one_hot_max_size | 64 | 0.945860 | -0.000030 | 誤差 |

**深さは既定(6)が最適で、増やすほど単調に悪化。** `one_hot_max_size` は完全に無反応だった。

> **注意**: CatBoost は `--save` を付けずに実行したため OOF が保存されておらず、**DeLong 検定は
> できていない**(AUC の比較のみ)。最良の d5 でも +0.00006 と基準(+0.00008)未満のため、
> 保存し直して検定する価値はないと判断した。

### 結論

**ハイパーパラメータの調整は、このプロジェクトでは打ち止め。** 効いたパラメータ調整は2件だけで、どちらも
「探索」ではなく **見落としの是正**だった。

| 効いた調整 | 効果 | 性質 |
|---|---|---|
| 収束確認(lr 引き下げ + early stopping) | +0.0008 | 特徴量を13→92列に増やしたのに木が100本のままだった |
| 列サブサンプリング(colsample 0.3 + depth 5) | +0.0002 | 全列を使うとどの木も年収TEを根に選び同質化していた |

外部調査の「列サブサンプリング導入後の追加調整はすべて -0.000023〜+0.000017 の誤差」という
報告と完全に一致した。**Optuna で同じ空間を広く探しても結果は変わらない**と判断する。

In [5]:
# 実行後にこのセルで結果を確認する
tuning.report()

              日時      モデル                軸                 試行名  OOF_AUC     基準との差    秒                               引数
2026-09-22 01:22 catboost            depth     catboost_hpo_d5  0.94595  0.000060 1242                        --depth 5
2026-09-21 14:39     lgbm       num_leaves       lgbm_hpo_nl63  0.94610  0.000005  733                  --num_leaves 63
2026-09-21 14:12     lgbm       num_leaves      lgbm_hpo_nl127  0.94610  0.000005  451                 --num_leaves 127
2026-09-21 14:47     lgbm       num_leaves      lgbm_hpo_nl127  0.94610  0.000005  507                 --num_leaves 127
2026-09-21 14:05     lgbm       num_leaves       lgbm_hpo_nl63  0.94610  0.000005  339                  --num_leaves 63
2026-09-22 00:52      xgb        subsample       xgb_hpo_sub08  0.94608  0.000003  560        --set-param subsample=0.8
2026-09-21 23:42      xgb        max_depth          xgb_hpo_d4  0.94608  0.000003  567          --set-param max_depth=4
2026-09-22 00:34      xgb min_child_weig

## 05-10. まとめ

**実行した 18 試行(計画 24 のうち LightGBM の 6 本は打ち切り)すべてが誤差または悪化で、採用はゼロだった。**

- LightGBM の `num_leaves` は `max_depth=5` が先に制約になっており、値を変えても木が変わらなかった
- XGBoost は深さを増やすと**有意に悪化**(z=-5.8 / -6.3)。現行の 5 が最適
- CatBoost も深さは既定(6)が最適で、増やすほど単調に悪化
- **パラメータが「未調整」に見えても、別のパラメータが先に制約になっていないか確認すべき**という教訓を得た

この結果は「効かなかった」という記録として `Log.md` に残してある。**同じ空間を Optuna で
探し直す価値はない**と判断する。